# HelicAI

### Predict and design the full electronic circular dichroism (ECD) spectrum of [6]helicene derivatives

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rguceda/HelicAI/blob/main/HelicAI.ipynb)

HelicAI is a neural-network surrogate of TD-DFT for substituted (*P*)-[6]helicenes. It predicts the complete ECD spectrum (250–600 nm) of a substituted [6]helicene in a fraction of a second, without any quantum-chemical calculation, 3D structure or geometry optimization. This notebook has two modules:

- **Module 1 — Predict your helicene.** Choose the substituent at each of the 16 positions of the scaffold and get the predicted ECD spectrum, with an applicability-domain check. Substituents outside the descriptor library can be used too: with all four descriptors for E2E-64, or with just their Hammett constant for E2E-H.
- **Module 2 — Design your helicene.** Define a design objective (for example, maximize or minimize a given ECD band, or shift it towards longer wavelengths), optionally with constraints, and let a genetic algorithm coupled to E2E-64 search a chemical space of billions of substitution patterns.

| Model | Inputs per position | Use it for |
|---|---|---|
| **E2E-64** | Hammett σ<sub>p</sub>, van der Waals volume *V*<sub>vdW</sub>, resonance constants σ<sub>R</sub><sup>+</sup> and σ<sub>R</sub><sup>−</sup> (64 inputs) | The most accurate predictions; substituents of the library, or new ones with all four descriptors. Used by Module 2. |
| **E2E-H** | Hammett σ<sub>p</sub> only (16 inputs) | Fast first-pass exploration: any σ<sub>p</sub> value can be placed at any position. |

Both models were trained on TD-DFT (M06/TZVP, CH<sub>2</sub>Cl<sub>2</sub>) spectra of 12,757 [6]helicenes bearing one to six substituents from a library of 16 groups. Spectra are given as *R*·10<sup>40</sup> / esu cm erg G<sup>−1</sup> on a 100-point grid, after Gaussian broadening of the transitions (σ = 0.30 eV). Full details are in the article and its Supporting Information.

**How to use it:** run the three setup cells once (*Runtime → Run all* also works), then edit the cells marked `EDIT THIS`. A CPU runtime is enough.

HelicAI is open source under the [MIT License](https://github.com/rguceda/HelicAI/blob/main/LICENSE). *If you use it, please cite:* [article reference to be added on publication].

---
## Setup
Run these cells once per session. The first one downloads the HelicAI repository (models, substituent library and scripts); the second installs anything missing and unpacks the two models; the third loads both models and the helper functions used by the two modules.

In [ ]:
#@title 1. Get HelicAI { display-mode: "form" }
import os, subprocess

REPO_URL = "https://github.com/rguceda/HelicAI.git"

if os.path.exists("HelicAI.ipynb") and os.path.isdir("scripts"):   # running inside a local copy
    BASE_DIR = os.getcwd()
else:                                                                 # Google Colab
    BASE_DIR = "/content/HelicAI"
    if not os.path.exists(BASE_DIR):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, BASE_DIR], check=True)
print("HelicAI folder:", BASE_DIR)

In [ ]:
#@title 2. Install and unpack the models { display-mode: "form" }
import sys, zipfile, importlib

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")       # keep TensorFlow quiet
SCRIPTS_DIR = os.path.join(BASE_DIR, "scripts")
DATA_DIR = os.path.join(BASE_DIR, "data")
MODELS_DIR = os.path.join(BASE_DIR, "models")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")           # spectra, figures and design results land here
if SCRIPTS_DIR not in sys.path:
    sys.path.append(SCRIPTS_DIR)

for module, pip_name in [("tensorflow", "tensorflow"), ("sklearn", "scikit-learn"), ("joblib", "joblib"),
                         ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
for label in ["E2E_64", "E2E_H"]:
    target = os.path.join(MODELS_DIR, label)
    if not os.path.exists(target):
        with zipfile.ZipFile(os.path.join(BASE_DIR, f"{label}_package.zip")) as z:
            z.extractall(target)
print("Models ready in", MODELS_DIR)

In [ ]:
#@title 3. Load both models and the helper functions { display-mode: "form" }
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from sklearn.exceptions import InconsistentVersionWarning
    warnings.filterwarnings("ignore", category=InconsistentVersionWarning)
except ImportError:
    pass

from predict_ecd import load_model_package, predict_ecd
import design_candidates as dc
from custom_objectives import band_objective, annotate_band_metrics

MODELS = {label: load_model_package(os.path.join(MODELS_DIR, label), label) for label in ("E2E_64", "E2E_H")}
LIBRARY_PATH = os.path.join(DATA_DIR, "substituent_library.json")
with open(LIBRARY_PATH) as f:
    LIBRARY = json.load(f)

PRETTY = {"E2E_64": "E2E-64", "E2E_H": "E2E-H"}
COLORS = {"E2E_64": "#c0272d", "E2E_H": "#2a78d6"}       # the colours used in the article
UNITS = r"$R\cdot10^{40}$ / esu cm erg G$^{-1}$"
N_POSITIONS = 16

# every name that identifies a library substituent: library names, aliases and the article's notation
_NAME_TO_KEY = {}
for key, name in LIBRARY["names"].items():
    for alias in [name, dc.display_name(name), dc.DISPLAY_NAMES.get(name, name)] + LIBRARY.get("aliases", {}).get(key, []):
        _NAME_TO_KEY.setdefault(alias.lower(), key)


def library_table():
    """The 16 substituents of the training library and their descriptors."""
    d = LIBRARY["descriptors"]
    rows = [{"name to type": LIBRARY["names"][k], "substituent": "–" + dc.display_name(LIBRARY["names"][k]),
             "σp": float(k), "VvdW / Å³": d["vdw"][k], "σR+": d["r_plus"][k], "σR−": d["r_minus"][k]}
            for k in sorted(LIBRARY["names"], key=float)]
    return pd.DataFrame(rows).set_index("name to type")


def descriptor_vector(compound: dict, label: str, new_substituents: dict = None) -> np.ndarray:
    """{position: substituent} -> the descriptor vector expected by the model.

    A substituent can be (i) the name of a library substituent ("Br", "NO2", "C≡CPh"...),
    (ii) the name of a substituent defined in new_substituents, e.g.
         {"CF3": {"hammett": 0.54, "vdw": 42.74, "r_plus": 0.23, "r_minus": 0.27}}
         (E2E-H only needs "hammett"), or
    (iii) for E2E-H only, a bare number, taken as its Hammett σp (e.g. {2: 0.17, 15: 0.29}).
    Unlisted positions carry hydrogen."""
    new_substituents = new_substituents or {}
    d, dflt = LIBRARY["descriptors"], LIBRARY["defaults"]
    hammett = np.full(N_POSITIONS, dflt["hammett"], dtype=float)
    vdw = np.full(N_POSITIONS, dflt["vdw"], dtype=float)
    r_plus = np.full(N_POSITIONS, dflt["r_plus"], dtype=float)
    r_minus = np.full(N_POSITIONS, dflt["r_minus"], dtype=float)
    for pos, sub in compound.items():
        pos = int(pos)
        if not 1 <= pos <= N_POSITIONS:
            raise ValueError(f"Position {pos} does not exist: positions run from 1 to 16.")
        i = pos - 1
        if isinstance(sub, (int, float)):
            if label != "E2E_H":
                raise ValueError(f"Position {pos}: a bare σ value ({sub}) can only be used with E2E-H. "
                                 "For E2E-64, define the substituent in new_substituents with its four descriptors.")
            hammett[i] = float(sub)
        elif str(sub).lower() in _NAME_TO_KEY:
            k = _NAME_TO_KEY[str(sub).lower()]
            hammett[i], vdw[i], r_plus[i], r_minus[i] = float(k), d["vdw"][k], d["r_plus"][k], d["r_minus"][k]
        elif sub in new_substituents:
            n = new_substituents[sub]
            hammett[i] = n["hammett"]
            if label == "E2E_64":
                missing = [x for x in ("vdw", "r_plus", "r_minus") if x not in n]
                if missing:
                    raise ValueError(f"'{sub}' lacks {missing}: E2E-64 needs all four descriptors (use E2E-H instead).")
                vdw[i], r_plus[i], r_minus[i] = n["vdw"], n["r_plus"], n["r_minus"]
        else:
            raise ValueError(f"'{sub}' is neither a library substituent nor defined in new_substituents. "
                             f"Library names: {sorted(LIBRARY['names'].values())}")
    return hammett if label == "E2E_H" else np.concatenate([hammett, vdw, r_plus, r_minus])


def describe(compound: dict) -> str:
    """{15: 'CP', 2: 'Br'} -> '2-Br, 15-C≡CPh'"""
    items = sorted((int(p), s) for p, s in compound.items())
    return ", ".join(f"σp({p}) = {s:g}" if isinstance(s, (int, float)) else f"{p}-{dc.display_name(str(s))}"
                     for p, s in items)


def predict(compound: dict, models=("E2E_64", "E2E_H"), new_substituents: dict = None, verbose=True) -> dict:
    """Predicts the ECD spectrum of one helicene with one or both models.
    Returns {label: (wavelengths, spectrum, ad_info)}."""
    results = {}
    for label in models:
        try:
            x = descriptor_vector(compound, label, new_substituents)
        except ValueError as err:
            if len(models) > 1:
                print(f"{PRETTY[label]} skipped: {err}")
                continue
            raise
        results[label] = predict_ecd(x, MODELS[label])
    if verbose:
        print(describe(compound))
        for label, (wl, spec, ad) in results.items():
            flag = "OUTSIDE the applicability domain -- extrapolation, use with caution" if ad["outside_reference_domain"] \
                else "inside the applicability domain"
            print(f"  {PRETTY[label]:7s} distance to nearest training molecule {ad['nearest_training_distance']:.2f} "
                  f"(threshold {ad['threshold']:.2f}): {flag}")
    return results


def plot_spectra(results: dict, title: str = "", ylim=None, save_as: str = None):
    """Overlays the predicted spectra (E2E-64 red, E2E-H blue)."""
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for label, (wl, spec, ad) in results.items():
        name = PRETTY[label] + ("  ⚠ outside AD" if ad["outside_reference_domain"] else "")
        ax.plot(wl, spec, color=COLORS[label], lw=2, label=name)
    ax.axhline(0, color="#8a8985", lw=0.8, zorder=0)
    ax.set_xlim(250, 600)
    if ylim:
        ax.set_ylim(-ylim, ylim)
    ax.set_xlabel(r"$\lambda$ / nm", fontsize=12)
    ax.set_ylabel(UNITS, fontsize=11)
    ax.set_title(title, fontsize=11, loc="left")
    ax.grid(color="#e4e3df", lw=0.6); ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    if save_as:
        path = os.path.join(OUTPUT_DIR, save_as)
        fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
        print("Figure saved:", path)
    plt.show()


def save_spectra(results: dict, filename: str) -> str:
    """Saves the predicted spectra as CSV (wavelength, one column per model)."""
    wl = next(iter(results.values()))[0]
    df = pd.DataFrame({"wavelength_nm": wl})
    for label, (_, spec, _) in results.items():
        df[f"{PRETTY[label]} (R·1e40 / esu cm erg G^-1)"] = spec
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print("Spectra saved:", path)
    return path


def safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_-]+", "_", text).strip("_")[:80]


print("Loaded:", ", ".join(f"{PRETTY[k]} ({len(v.feature_columns)} inputs)" for k, v in MODELS.items()))
print("Ready. Outputs are written to", OUTPUT_DIR)

---
# Module 1 — Predict your helicene

Each of the 16 positions of the (*P*)-[6]helicene scaffold either carries a substituent or a hydrogen. Positions *i* and 17 − *i* are related by the *C*<sub>2</sub> axis, so, for example, 2,15- and 15,2-disubstitution are the same molecule.

The models do not see atoms, bonds or 3D coordinates: each substituent enters through a few descriptors (σ<sub>p</sub> only for E2E-H; σ<sub>p</sub>, *V*<sub>vdW</sub>, σ<sub>R</sub><sup>+</sup> and σ<sub>R</sub><sup>−</sup> for E2E-64), and the whole 100-point spectrum is predicted in a single step. This is also why substituents absent from the training library can be explored, as long as their descriptors are known.

These are the 16 substituents of the training library; type them by the name in the first column (the article's notation, e.g. `C≡CPh`, is also accepted):

In [ ]:
library_table()

## 1.1 Predict a helicene

Describe your molecule as `{position: substituent}`. Unlisted positions carry hydrogen. The cell predicts the spectrum with both models, reports the applicability domain (Section 1.3) and saves the spectra (CSV) and the figure (PNG) in the `outputs` folder.

In [ ]:
# ---- EDIT THIS ---------------------------------------------------------------
compound = {2: "Br", 15: "Br"}          # 2,15-dibromo[6]helicene
# --------------------------------------------------------------------------------

spectra = predict(compound)
name = safe_name(describe(compound))
plot_spectra(spectra, title=describe(compound), save_as=f"prediction_{name}.png")
save_spectra(spectra, f"prediction_{name}.csv");

## 1.2 Substituents outside the descriptor library

- **E2E-64** needs the four descriptors of the new substituent: σ<sub>p</sub>, *V*<sub>vdW</sub> (Å<sup>3</sup>), σ<sub>R</sub><sup>+</sup> and σ<sub>R</sub><sup>−</sup>.
- **E2E-H** only needs its Hammett σ<sub>p</sub>; you can even give a bare σ value at any position.

`data/new_substituent_library_EXAMPLE.json` contains the four new groups used to validate HelicAI in the article (–CF<sub>3</sub>, –CONH<sub>2</sub>, –CH<sub>2</sub>CH<sub>3</sub> and –OCH<sub>2</sub>CH<sub>3</sub>):

In [ ]:
with open(os.path.join(DATA_DIR, "new_substituent_library_EXAMPLE.json")) as f:
    _example = json.load(f)
def _short(name):                        # "Am (CONH2)" -> "CONH2"
    m = re.search(r"\((.*)\)", name)
    return m.group(1) if m else name

NEW_SUBSTITUENTS = {
    _short(_example["names"][k]): {"hammett": float(k), "vdw": _example["descriptors"]["vdw"][k],
                                         "r_plus": _example["descriptors"]["r_plus"][k],
                                         "r_minus": _example["descriptors"]["r_minus"][k]}
    for k in _example["names"]
}
pd.DataFrame(NEW_SUBSTITUENTS).T

In [ ]:
# ---- EDIT THIS: a helicene with never-seen substituents (both models) ------------------------
compound_new = {2: "CF3", 15: "CF3"}
# you can add your own substituent here, e.g.
# NEW_SUBSTITUENTS["MyGroup"] = {"hammett": 0.00, "vdw": 0.00, "r_plus": 0.00, "r_minus": 0.00}   # <- your values
# ------------------------------------------------------------------------------------------------

spectra_new = predict(compound_new, new_substituents=NEW_SUBSTITUENTS)
plot_spectra(spectra_new, title=describe(compound_new) + " (substituent not in the training set)")

With **E2E-H** any Hammett constant can be placed at any position, which makes it a first-pass tool to explore electronic effects. The cell below predicts the helicene of Figure 6 (bottom) of the article, with –SiMe<sub>2</sub>F (σ<sub>p</sub> = 0.17) at position 2 and –CBr<sub>3</sub> (σ<sub>p</sub> = 0.29) at position 15. σ<sub>p</sub> values for hundreds of substituents are tabulated in Hansch, Leo and Taft, *Chem. Rev.* **1991**, *91*, 165–195. Use the *para* constant (σ<sub>p</sub>), not σ<sub>m</sub>.

In [ ]:
# ---- EDIT THIS: Hammett constants only (E2E-H) ------------------------------------------------
compound_sigma = {2: 0.17, 15: 0.29}      # -SiMe2F at 2, -CBr3 at 15
# ------------------------------------------------------------------------------------------------

spectra_sigma = predict(compound_sigma, models=("E2E_H",))
plot_spectra(spectra_sigma, title=describe(compound_sigma))

## 1.3 The applicability-domain (AD) warning

Every prediction is accompanied by the distance, in the space of standardized descriptors, from your molecule to its nearest neighbour in the training set. The threshold is the 95th percentile of the same distance among training molecules (τ = 3.08 for E2E-H and 6.72 for E2E-64).

- **Inside the AD:** your molecule is about as close to the training data as a typical training molecule is to its own neighbours.
- **⚠ Outside the AD:** the model is extrapolating (for example, many substituents, or descriptor values far from those of the library). The prediction is still returned, but should be interpreted with caution and, ideally, confirmed by TD-DFT.

This is a heuristic reliability flag, not a hard cutoff.

---
# Module 2 — Design your helicene

Inverse design couples E2E-64 with a genetic algorithm (GA), exactly as in Case IV of the article (Supporting Information, Section S6):

1. **Encoding.** Each candidate is a genome of 16 genes, one per position, each empty (H) or holding a substituent (8 genes, one per mirror pair, when *C*<sub>2</sub> symmetry is imposed).
2. **Evaluation.** The spectrum of every new molecule is predicted with E2E-64 and scored with the objective; *C*<sub>2</sub>-equivalent molecules are evaluated only once.
3. **Evolution.** The best 10% passes unchanged to the next generation (elitism); the rest is produced by tournament selection (size 4), uniform crossover and mutation (probability 0.15 per gene), and repaired to keep the number of substituents within [*k*<sub>min</sub>, *k*<sub>max</sub>].
4. **AD penalty.** The score of candidates outside the applicability domain is down-weighted by 1 / (1 + *w*·max[0, (*d* − τ)/τ]), with *w* = 1.0, so the search avoids unreliable extrapolations.

The article used a population of 1,000 individuals evolved over 50 generations (random seed 42). A run takes about half a minute on a CPU.

### Design objectives

The objective is built with `band_objective(metric, direction)`:

| `metric` | Quantity optimized |
|---|---|
| `"positive_band"` | area of the largest positive band, *A*<sub>max</sub><sup>+</sup> |
| `"negative_band"` | area of the largest negative band, \|*A*<sub>max</sub><sup>−</sup>\| |
| `"peak_height"` | height of the most intense peak or valley, of either sign |
| `"positive_peak"` | height of the most intense positive peak |
| `"negative_peak"` | depth of the most intense negative valley |

`direction="max"` searches for the largest value, `direction="min"` for the smallest one (internally the GA then maximizes 1/(1 + value), so the `score` column is not in physical units). Band areas are always reported in physical units (*R*·10<sup>40</sup> / esu cm erg G<sup>−1</sup> × nm) in the columns `A+max` and `|A-max|`.

Setting `target_nm` evaluates the objective only within `target_nm ± window_nm` (e.g. a positive band around 500 nm, to red-shift the response). You can also restrict the substituents (`allowed_substituents`), the positions (`allowed_positions`) and impose *C*<sub>2</sub> symmetry (`force_symmetric=True`).

## 2.1 Your design

Choose what to optimize and run the cell. The six design subcases of the article (Supporting Information, Table S19) are obtained by changing only `metric`, `direction`, `k_max` and `target_nm`; their exact settings are listed in the [README](https://github.com/rguceda/HelicAI#inverse-design-as-in-the-article).

In [ ]:
# ============================================================================================
# EDIT THIS: what to optimize
#   metric:    "positive_band", "negative_band", "peak_height", "positive_peak", "negative_peak"
#   direction: "max" (as large as possible) or "min" (as small as possible)
# ============================================================================================
my_objective = band_objective(metric="positive_band", direction="max")

designs = dc.genetic_search(
    model_dir=os.path.join(MODELS_DIR, "E2E_64"),
    label="E2E_64",
    library_path=LIBRARY_PATH,
    objective=my_objective,           # a function, not a string
    target_nm=False,                  # e.g. 500 to optimize only within 500 ± window_nm
    window_nm=30,
    ad_penalty_weight=1.0,            # 0 = no applicability-domain penalty
    allowed_substituents=False,       # e.g. ["F", "Br", "NO2", "C≡CPh"]
    allowed_positions=False,          # e.g. [1, 2, 3, 4, 13, 14, 15, 16]
    force_symmetric=False,            # True = only C2-symmetric molecules
    k_min=1,
    k_max=4,                          # number of substituents (the training set had up to 6)
    population_size=1000,
    n_generations=50,
    seed=42,
    output_dir=os.path.join(OUTPUT_DIR, "my_design"),
    top_n=0,                          # the table is shown below instead
    plot_top_n=10,
)

# band areas in physical units (R·10^40 / esu cm erg G^-1 × nm), whatever objective/direction you used
designs = annotate_band_metrics(designs)
dc.summary_table(designs, 10)

## 2.2 Constrained designs

Constraints narrow the search to what you can actually make. The example below looks for *C*<sub>2</sub>-symmetric helicenes with the largest positive band, using only halogens, –NO<sub>2</sub> and –C≡C–Ph, at most four substituents (two mirror pairs). Any of the options of Section 2.1 can be combined in the same way.

In [ ]:
# ---- EDIT THIS ---------------------------------------------------------------
constrained = dc.genetic_search(
    model_dir=os.path.join(MODELS_DIR, "E2E_64"), label="E2E_64", library_path=LIBRARY_PATH,
    objective=band_objective("positive_band", "max"),
    allowed_substituents=["F", "Cl", "Br", "I", "NO2", "C≡CPh"],
    allowed_positions=False,
    force_symmetric=True,
    k_min=2, k_max=4,
    ad_penalty_weight=1.0, population_size=1000, n_generations=50, seed=42,
    output_dir=os.path.join(OUTPUT_DIR, "constrained_design"), top_n=0, plot_top_n=6,
)
# --------------------------------------------------------------------------------
constrained = annotate_band_metrics(constrained)
dc.summary_table(constrained, 6)

---
## Download your results

Every prediction (CSV and PNG) and every design run (`*_genetic_results.csv`, with the substitution pattern, scores, band areas and AD flags of all evaluated molecules) is saved in the `outputs` folder. In Colab, the cell below packs it into a zip file and downloads it.

In [ ]:
import shutil
archive = shutil.make_archive(os.path.join(BASE_DIR, "HelicAI_outputs"), "zip", OUTPUT_DIR)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Results packed in:", archive)

---
## Going further

All the scripts used in the article are in `scripts/`, each with its own documentation and command-line interface:

- `predict_ecd.py` — load a model package and predict one spectrum (the core of Module 1).
- `design_candidates.py` and `custom_objectives.py` — genetic algorithm, hill climbing and random search, with the band objectives of the article (Module 2).
- `screen_substituents.py`, `band_area_scaling_study.py`, `spectrum_scan.py` — exhaustive and sampled screening of whole substitution levels (Case II).
- `validate_new_substituents.py`, `validate_new_substituents_progression.py` — validation against TD-DFT spectra of helicenes with new substituents (Case III).
- `train_pipeline.py`, `characterize_model.py`, `baseline_comparison.py` — training, evaluation and baselines of the models. The training dataset is `data/ECD_total_Gauss.csv`.